> **Portfolio version:** Confidential source data, local computer paths, server names, and saved notebook outputs have been removed or replaced with placeholders. The original operational datasets are not included.


# FPSO UARU NDT Operations Analytics
### Python ETL - Sprint 1: Create Staging Table

# 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

# 2. Define Source File Paths

In [ ]:

piping_file = "../01_Data/sample_piping_ndt.xlsx"

structure_file = "../01_Data/sample_structure_ndt.xlsx"

pmi_file = "../01_Data/sample_pmi_summary.xlsx"



# 3. Verify Source Workbooks

In [ ]:
pd.ExcelFile(piping_file).sheet_names


In [ ]:
pd.ExcelFile(structure_file).sheet_names


In [ ]:
pd.ExcelFile(pmi_file).sheet_names

# 4. Load Source Data
**Objective**

Load the required worksheets from the source Excel workbooks into Pandas DataFrames for inspection and subsequent ETL processing.

# Read PIPING sheet

In [ ]:
piping_df = pd.read_excel(
    piping_file,
    sheet_name="PIPING",
    header=11
)

piping_df.shape

# Read PMI sheet

In [ ]:
pmi_df = pd.read_excel(
    piping_file,
    sheet_name="PMI",
    header=11
)

pmi_df.shape

# Read PAUT sheet

In [ ]:
paut_df = pd.read_excel(
    piping_file,
    sheet_name="PAUT",
    header=11
)

paut_df.shape

# Read Structure sheet

In [ ]:
structure_df = pd.read_excel(
    structure_file,
    sheet_name="STR",
    header=11
)

structure_df.shape

# Read Separate PMI workbook

In [ ]:
pmi_summary_df = pd.read_excel(
    pmi_file,
    sheet_name="PMI",
    header=11
)

pmi_summary_df.shape

# 5. Inspect Source Data Structure

**Objective**

Review the shape, column names, and first few rows of each source sheet before cleaning and standardizing the data.

In [ ]:
print("PIPING shape:", piping_df.shape)
print("PMI shape:", pmi_df.shape)
print("PAUT shape:", paut_df.shape)
print("STRUCTURE shape:", structure_df.shape)
print("PMI Summary shape:", pmi_summary_df.shape)

In [ ]:
piping_df.columns

In [ ]:
pmi_df.columns

In [ ]:
paut_df.columns

In [ ]:
structure_df.columns

In [ ]:
pmi_summary_df.columns

# 6. Standardize Column Names

**Objective**

Standardize column names from all source worksheets into a common format before data cleaning and appending. This ensures that all DataFrames share the same schema for subsequent ETL processing.

In [ ]:
# =====================================
# Step 6 - Standardize Column Names
# =====================================

column_mapping = {
    
    # Date Columns
    "Req Date\n(DD/MM/YYYY)": "RequestDate",
    "JDR Date\n(DD/MM/YYYY)": "JDRDate",

    # Master Data
    "Module": "Module",
    "Test \nMethod": "Method",
    "Request No.": "RequestNo",

    # Quantity Columns
    "Requested\nJts": "RequestedJts",
    "Requested\nLength\nMtr": "RequestedMtr",
    "Requested\nPoints": "RequestedPoints",
    "JDR\n(Films / Jts / Mtr / Pts)": "JDRQty",
    "NDT completed Qty (Jts)": "CompletedQty",

    # Reference Columns
    "JOB DONE REPORT NO": "JDRNo",
    "Job done Report No.": "JDRNo",
    "NDT Report No.": "NDTReportNo",
    "Job Verification Report No \n(JVR No)": "JVRNo",

    # Columns to be removed later
    "S.No.": "SNo",
    "Period\n(For Fortnight Reporting)": "Period",
    "Sub-con": "SubCon",
    "Location": "Location",
    "Remarks": "Remarks",
    "RECEIVED": "Received",
    "Unnamed: 16": "BlankCol1",
    "Unnamed: 18": "BlankCol2"
}

In [ ]:
# =====================================
# Rename Columns
# =====================================

def standardize_columns(df):
    return df.rename(columns=column_mapping)

In [ ]:
piping_df = standardize_columns(piping_df)
pmi_df = standardize_columns(pmi_df)
paut_df = standardize_columns(paut_df)
structure_df = standardize_columns(structure_df)
pmi_summary_df = standardize_columns(pmi_summary_df)

In [ ]:
piping_df.columns

# 7. Remove Unwanted Columns

**Objective**

Remove source columns that are not required for analysis, reporting, or the final star schema model.

In [ ]:
# =====================================
# Step 7 - Remove Unwanted Columns
# =====================================

drop_columns = [
    "SNo",
    "Period",
    "SubCon",
    "Location",
    "JDRNo",
    "NDTReportNo",
    "JVRNo",
    "Remarks",
    "Received",
    "BlankCol1",
    "BlankCol2"
]

def drop_unwanted_columns(df):
    return df.drop(columns=[col for col in drop_columns if col in df.columns])

In [ ]:
piping_df = drop_unwanted_columns(piping_df)
pmi_df = drop_unwanted_columns(pmi_df)
paut_df = drop_unwanted_columns(paut_df)
structure_df = drop_unwanted_columns(structure_df)
pmi_summary_df = drop_unwanted_columns(pmi_summary_df)

In [ ]:
piping_df.columns

In [ ]:
print("PIPING:", piping_df.shape)
print("PMI:", pmi_df.shape)
print("PAUT:", paut_df.shape)
print("STRUCTURE:", structure_df.shape)
print("PMI Summary:", pmi_summary_df.shape)

# 8. Create Standard Requested Quantity Column

**Objective**

Combine the different requested quantity columns into one standard column called `RequestedQty`, so all sheets can be appended into one staging table.

In [ ]:
# =====================================
# Step 8 - Create RequestedQty
# =====================================

def standardize_requested_quantity(df):

    df = df.copy()

    # Create missing columns
    for col in ["RequestedJts", "RequestedMtr", "RequestedPoints"]:
        if col not in df.columns:
            df[col] = np.nan

    # Create RequestedQty
    df["RequestedQty"] = df["RequestedJts"]

    df["RequestedQty"] = df["RequestedQty"].fillna(df["RequestedMtr"])
    df["RequestedQty"] = df["RequestedQty"].fillna(df["RequestedPoints"])

    return df

In [ ]:
# =====================================
# Apply to all DataFrames
# =====================================

piping_df = standardize_requested_quantity(piping_df)
pmi_df = standardize_requested_quantity(pmi_df)
paut_df = standardize_requested_quantity(paut_df)
structure_df = standardize_requested_quantity(structure_df)
pmi_summary_df = standardize_requested_quantity(pmi_summary_df)

In [ ]:
# =====================================
# Step 9 - Add Discipline Column
# =====================================

piping_df["Discipline"] = "Piping"
pmi_df["Discipline"] = "Piping"
paut_df["Discipline"] = "Piping"
structure_df["Discipline"] = "Structure"
pmi_summary_df["Discipline"] = "Piping"

In [ ]:
print(piping_df["Discipline"].unique())
print(structure_df["Discipline"].unique())

# 10. Create Consolidated Staging Table

**Objective**

Select the final standardized columns from each source DataFrame and append all sheets into one consolidated staging table.

In [ ]:
# =====================================
# Step 10 - Create Consolidated Staging Table
# =====================================

staging_columns = [
    "RequestDate",
    "JDRDate",
    "Discipline",
    "Module",
    "Method",
    "RequestNo",
    "RequestedQty",
    "CompletedQty",
    "JDRQty"
]

In [ ]:
piping_clean = piping_df[staging_columns]
pmi_clean = pmi_df[staging_columns]
paut_clean = paut_df[staging_columns]
structure_clean = structure_df[staging_columns]
pmi_summary_clean = pmi_summary_df[staging_columns]

In [ ]:
stg_ndt_inspection = pd.concat(
    [
        piping_clean,
        pmi_clean,
        paut_clean,
        structure_clean,
        pmi_summary_clean
    ],
    ignore_index=True
)

In [ ]:
stg_ndt_inspection.shape

In [ ]:
stg_ndt_inspection.head()

In [ ]:
# Data types
stg_ndt_inspection.info()

In [ ]:
# Missing values
stg_ndt_inspection.isnull().sum()

In [ ]:
# First few rows
stg_ndt_inspection.head()

In [ ]:
stg_ndt_inspection[stg_ndt_inspection["RequestDate"].isna()]

In [ ]:
stg_ndt_inspection[stg_ndt_inspection["RequestNo"].isna()]

In [ ]:
stg_ndt_inspection[stg_ndt_inspection["RequestedQty"].isna()]

# =====================================
# Step 11 - Remove Rows Without Inspection Quantities
# =====================================

In [ ]:
stg_ndt_inspection = stg_ndt_inspection.dropna(
    subset=["RequestedQty", "CompletedQty", "JDRQty"],
    how="all"
)

print(stg_ndt_inspection.shape)

## Step 11.1 — Clean and Convert the Date Columns

In [ ]:
# =====================================
# Business Rule
# Correct known source-system date entry errors
# before datetime conversion.
# =====================================

# =====================================
# Step 11.1 - Clean and Convert Date Values
# =====================================

import re
from datetime import datetime


def clean_ndt_date(value):
    """
    Clean and convert mixed NDT date values.
    """

    # Keep missing values as missing
    if pd.isna(value):
        return pd.NaT

    # Preserve existing datetime values
    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.Timestamp(value)

    # Handle numeric values
    if isinstance(value, (int, float)):

        text_value = str(value).strip()

        # Handle decimal-style source dates
        # 1305.2024 -> 13.05.2024
        # 12.102023 -> 12.10.2023
        # 8.062024  -> 08.06.2024

        if "." in text_value:
            left_part, right_part = text_value.split(".", 1)
            digits_only = left_part + right_part

            if digits_only.isdigit() and len(digits_only) in [7, 8]:
                parsed_date = pd.to_datetime(
                    digits_only,
                    format="%d%m%Y",
                    errors="coerce"
                )

                if pd.notna(parsed_date):
                    return parsed_date

        # Handle genuine Excel serial dates
        if 30000 <= float(value) <= 60000:
            return pd.to_datetime(
                value,
                unit="D",
                origin="1899-12-30",
                errors="coerce"
            )

        return pd.NaT

    # Convert to a cleaned string
    cleaned_value = str(value).strip()

    # Standardize separators
    cleaned_value = cleaned_value.replace(",", ".")
    cleaned_value = cleaned_value.replace("/", ".")

    # Remove spaces and repeated dots
    cleaned_value = re.sub(r"\s+", "", cleaned_value)
    cleaned_value = re.sub(r"\.+", ".", cleaned_value)

    # Correct known malformed years
    cleaned_value = re.sub(r"\.224$", ".2024", cleaned_value)
    cleaned_value = re.sub(r"\.20324$", ".2024", cleaned_value)

    known_date_corrections = {
        "31.09.2024": "30.09.2024",
        "31.11.2023": "30.11.2023",
        "31.04.2024": "30.04.2024",
        "058.02.2024": "05.02.2024",
        "8.11.202": "08.11.2024",
        "29.05.204": "29.05.2024",
        "30.05.204": "30.05.2024",
        "28.09.204": "28.09.2024",
        "23.02.20224": "23.02.2024",
        "13.02.02024": "13.02.2024",
        "16.02.2025`": "16.02.2025",
        "I9.04.2024": "19.04.2024",
        "I3.04.2024": "13.04.2024",
        "I5.04.2024": "15.04.2024",
        "I7.04.2024": "17.04.2024",
        "I8.04.2024": "18.04.2024"
    }

    cleaned_value = known_date_corrections.get(
        cleaned_value,
        cleaned_value
    )

    # Convert DD.MM.YYYY to datetime
    return pd.to_datetime(
        cleaned_value,
        format="%d.%m.%Y",
        errors="coerce"
    )


# =====================================
# Preserve Raw Date Values
# =====================================

stg_ndt_inspection["RequestDateRaw"] = (
    stg_ndt_inspection["RequestDate"].copy()
)

stg_ndt_inspection["JDRDateRaw"] = (
    stg_ndt_inspection["JDRDate"].copy()
)


# Apply date cleaning
stg_ndt_inspection["RequestDate"] = (
    stg_ndt_inspection["RequestDate"]
    .apply(clean_ndt_date)
)

stg_ndt_inspection["JDRDate"] = (
    stg_ndt_inspection["JDRDate"]
    .apply(clean_ndt_date)
)


print("=" * 60)
print("DATE CLEANING RESULTS")
print("=" * 60)

print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].dtypes
)

print("\nMissing values after date cleaning:")

print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].isna().sum()
)

# 12. Handle Missing Dates

**Objective**

Fill missing RequestDate and JDRDate values using the available date from the same inspection record where possible.

In [ ]:
# =====================================
# Step 12 - Handle Missing Dates
# =====================================

# Fill missing RequestDate using JDRDate
stg_ndt_inspection["RequestDate"] = stg_ndt_inspection["RequestDate"].fillna(
    stg_ndt_inspection["JDRDate"]
)

# Fill missing JDRDate using RequestDate
stg_ndt_inspection["JDRDate"] = stg_ndt_inspection["JDRDate"].fillna(
    stg_ndt_inspection["RequestDate"]
)

# Check remaining missing dates
stg_ndt_inspection[["RequestDate", "JDRDate"]].isnull().sum()

In [ ]:
# =====================================
# Business Rule
# Remove records where both RequestDate and JDRDate
# are missing because they cannot participate in
# time-based analysis.
# =====================================

# =====================================
# Step 12.1 - Remove Records Without Both Dates
# =====================================

stg_ndt_inspection = stg_ndt_inspection.loc[
    ~(
        stg_ndt_inspection["RequestDate"].isna()
        &
        stg_ndt_inspection["JDRDate"].isna()
    )
].reset_index(drop=True)

print("Remaining rows:", len(stg_ndt_inspection))

print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].isna().sum()
)

# 13. Convert Data Types

**Objective**

Convert columns to appropriate data types to ensure accurate analysis, calculations, and loading into SQL and Power BI.

In [ ]:

# =====================================
# Step 13 - Convert Final Data Types
# =====================================

# Convert text columns
text_columns = [
    "Discipline",
    "Module",
    "Method",
    "RequestNo"
]

for col in text_columns:
    stg_ndt_inspection[col] = (
        stg_ndt_inspection[col]
        .astype("string")
    )

# Convert numeric columns
numeric_columns = [
    "RequestedQty",
    "CompletedQty",
    "JDRQty"
]

for col in numeric_columns:
    stg_ndt_inspection[col] = pd.to_numeric(
        stg_ndt_inspection[col],
        errors="coerce"
    )

print("=" * 60)
print("FINAL DATA TYPES")
print("=" * 60)

print(stg_ndt_inspection.dtypes)


In [ ]:
print(stg_ndt_inspection[["RequestDate", "JDRDate"]].dtypes)

print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].isna().sum()
)

In [ ]:
stg_ndt_inspection.info()

# 14. Create InspectionID

**Objective**

Create a unique surrogate key for each inspection activity record because `RequestNo` is not unique.

In [ ]:
stg_ndt_inspection.insert(
    0,
    "InspectionID",
    range(1, len(stg_ndt_inspection) + 1)
)

In [ ]:
stg_ndt_inspection.info()

In [ ]:
stg_ndt_inspection.head()

In [ ]:
# =====================================
# Step 14.1 - Correct Known Malformed Dates
# =====================================

date_corrections = {
    # Incorrect year 2204
    63: {
        "RequestDate": "2024-03-09",
        "JDRDate": "2024-03-09"
    },
    64: {
        "RequestDate": "2024-03-09",
        "JDRDate": "2024-03-09"
    },

    # Incorrect year 2027
    2471: {
        "RequestDate": "2024-08-03",
        "JDRDate": "2024-08-03"
    },
    2472: {
        "RequestDate": "2024-08-03",
        "JDRDate": "2024-08-03"
    },

    # Request date is correct; only JDR date is incorrect
    3480: {
        "JDRDate": "2024-08-28"
    },

    # Incorrect 2014 / neighbouring duplicate dates
    10815: {
        "RequestDate": "2024-01-18",
        "JDRDate": "2024-01-18"
    },
    10816: {
        "RequestDate": "2024-01-18",
        "JDRDate": "2024-01-18"
    },

    # Incorrect 2027 / neighbouring duplicate dates
    11127: {
        "RequestDate": "2024-02-07",
        "JDRDate": "2024-02-07"
    },
    11128: {
        "RequestDate": "2024-02-07",
        "JDRDate": "2024-02-07"
    },

    # Incorrect sequential years 2025–2029
    12416: {
        "RequestDate": "2024-05-16",
        "JDRDate": "2024-05-16"
    },
    12417: {
        "RequestDate": "2024-05-16",
        "JDRDate": "2024-05-16"
    },
    12418: {
        "RequestDate": "2024-05-16",
        "JDRDate": "2024-05-16"
    },
    12419: {
        "RequestDate": "2024-05-16",
        "JDRDate": "2024-05-16"
    },
    12420: {
        "RequestDate": "2024-05-16",
        "JDRDate": "2024-05-16"
    }
}

for inspection_id, corrections in date_corrections.items():
    mask = stg_ndt_inspection["InspectionID"] == inspection_id

    if "RequestDate" in corrections:
        stg_ndt_inspection.loc[mask, "RequestDate"] = pd.Timestamp(
            corrections["RequestDate"]
        )

    if "JDRDate" in corrections:
        stg_ndt_inspection.loc[mask, "JDRDate"] = pd.Timestamp(
            corrections["JDRDate"]
        )

print("=" * 60)
print("CORRECTED DATE RECORDS")
print("=" * 60)

print(
    stg_ndt_inspection.loc[
        stg_ndt_inspection["InspectionID"].isin(date_corrections.keys()),
        [
            "InspectionID",
            "RequestNo",
            "RequestDate",
            "JDRDate"
        ]
    ].sort_values("InspectionID").to_string(index=False)
)

In [ ]:
# =====================================
# Step 14.2 - Remove Confirmed Future-Dated Records
# =====================================

invalid_future_ids = [2470, 3479]

stg_ndt_inspection = stg_ndt_inspection[
    ~stg_ndt_inspection["InspectionID"].isin(invalid_future_ids)
].copy()

print("Rows after removing confirmed future-dated records:")
print(stg_ndt_inspection.shape)
print(stg_ndt_inspection["InspectionID"].isin([2470, 3479]).sum())

In [ ]:
# ==========================================
# Step 14.3 - Validate Project Date Range
# ==========================================

PROJECT_START_DATE = pd.Timestamp("2023-01-01")
PROJECT_END_DATE = pd.Timestamp("2026-12-31")

request_date_invalid = (
    stg_ndt_inspection["RequestDate"].notna()
    & ~stg_ndt_inspection["RequestDate"].between(
        PROJECT_START_DATE,
        PROJECT_END_DATE
    )
)

jdr_date_invalid = (
    stg_ndt_inspection["JDRDate"].notna()
    & ~stg_ndt_inspection["JDRDate"].between(
        PROJECT_START_DATE,
        PROJECT_END_DATE
    )
)

invalid_project_dates = stg_ndt_inspection.loc[
    request_date_invalid | jdr_date_invalid,
    [
        "InspectionID",
        "RequestNo",
        "RequestDateRaw",
        "JDRDateRaw",
        "RequestDate",
        "JDRDate"
    ]
].copy()

print("=" * 60)
print("PROJECT DATE RANGE VALIDATION")
print("=" * 60)

if invalid_project_dates.empty:
    print("All dates fall within the expected project range.")
else:
    print(
        invalid_project_dates.to_string(index=False)
    )

    print(
        "\nNumber of records outside project date range:",
        len(invalid_project_dates)
    )

# 15. Data Quality Validation

**Objective**

Validate the staging table to ensure the ETL process has produced a clean and consistent dataset before saving.

In [ ]:
print("=" * 60)
print("STAGING TABLE SHAPE")
print("=" * 60)

print(stg_ndt_inspection.shape)

In [ ]:
print("=" * 60)
print("DATA TYPES")
print("=" * 60)

stg_ndt_inspection.info()

In [ ]:
print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

stg_ndt_inspection.isnull().sum()

In [ ]:
print("=" * 60)
print("INSPECTION ID UNIQUE?")
print("=" * 60)

stg_ndt_inspection["InspectionID"].is_unique

In [ ]:
print("=" * 60)
print("UNIQUE REQUESTS")
print("=" * 60)

print("Total Rows:", len(stg_ndt_inspection))
print("Unique Request Numbers:", stg_ndt_inspection["RequestNo"].nunique())

## Step 15.1 - Remaining Data Quality Issues

In [ ]:
remaining_missing = stg_ndt_inspection.loc[
    stg_ndt_inspection["RequestDate"].isna()
    &
    stg_ndt_inspection["JDRDate"].isna()
]

display(remaining_missing)

# 16. Save Staging Table

**Objective**

Export the cleaned staging table for use in the data warehouse model.

In [ ]:
stg_ndt_inspection.to_csv(
    "../01_Data/processed/stg_ndt_inspection.csv",
    index=False
)

print("Staging table saved successfully.")

In [ ]:
print("=" * 60)
print("STAGING TABLE EXPORTED")
print("=" * 60)

print("Rows:", len(stg_ndt_inspection))
print("Columns:", len(stg_ndt_inspection.columns))

print("\nMissing Dates:")
print(
    stg_ndt_inspection[
        ["RequestDate", "JDRDate"]
    ].isna().sum()
)